# Tripitaka PDF Downloader from IFBC

This notebook scrapes the IFBC website (https://download.ifbcnet.org/) to download all Tripitaka books systematically.

**Website Structure:**
- Main category page: Lists all Tripitaka sections
- Each section page: Contains Google Drive links to PDFs
- PDFs are organized by section (e.g., Digha Nikaya, Majjhima Nikaya, etc.)

**Output Structure:**
```
{project_dir}/data/raw/tripitaka/
    ├── tripitaka-digha-nikaya/
    │   ├── book1.pdf
    │   └── book2.pdf
    ├── tripitaka-majjhima-nikaya/
    └── ...
```

## 1. Setup and Imports

In [1]:
# Install required packages
!pip install -q requests beautifulsoup4 gdown tqdm lxml

In [2]:
import requests
from bs4 import BeautifulSoup
import gdown
import os
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs
from tqdm.auto import tqdm
import json
from datetime import datetime

## 2. Configuration

In [3]:
# Mount Google Drive (if running on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
except:
    IS_COLAB = False
    print("Not running on Colab")

Mounted at /content/drive


In [4]:
# Configuration
BASE_URL = "https://download.ifbcnet.org/category/thripitaka-books/"
PROJECT_DIR = "/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/"
TRIPITAKA_DIR = os.path.join(PROJECT_DIR, "data/raw/tripitaka")

# Create base directory if it doesn't exist
os.makedirs(TRIPITAKA_DIR, exist_ok=True)

# Request headers to mimic a browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print(f"Tripitaka PDFs will be saved to: {TRIPITAKA_DIR}")

Tripitaka PDFs will be saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka


## 3. Helper Functions

In [5]:
def clean_section_name(url):
    """
    Extract clean section name from URL or title.
    Example: 'tripitaka-digha-nikaya' from URL
    """
    # Try to extract from URL path
    path = urlparse(url).path
    # Remove leading/trailing slashes and dates
    parts = [p for p in path.split('/') if p and not re.match(r'^\d{4}$', p)]

    if parts:
        # Get the last meaningful part (usually the slug)
        section_name = parts[-1]
        # Remove URL encoding artifacts
        section_name = re.sub(r'%[0-9a-fA-F]{2}', '', section_name)
        # Clean up the name
        section_name = re.sub(r'[^a-zA-Z0-9-_]', '', section_name)
        return section_name
    return "unknown-section"

def extract_google_drive_id(url):
    """
    Extract file ID from various Google Drive URL formats.
    """
    patterns = [
        r'drive\.google\.com/file/d/([a-zA-Z0-9_-]+)',
        r'drive\.google\.com/open\?id=([a-zA-Z0-9_-]+)',
        r'/d/([a-zA-Z0-9_-]+)',
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None

def download_from_google_drive(file_id, output_path):
    """
    Download a file from Google Drive using gdown.
    """
    try:
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, output_path, quiet=False)
        return True
    except Exception as e:
        print(f"Error downloading {file_id}: {str(e)}")
        return False

def sanitize_filename(filename):
    """
    Clean filename for safe storage.
    """
    # Remove or replace invalid characters
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    # Remove leading/trailing spaces and dots
    filename = filename.strip('. ')
    return filename

## 4. Web Scraping Functions

In [6]:
def get_section_links(category_url, max_pages=10):
    """
    Scrape all section links from the category page(s).
    Handles pagination if present.
    """
    section_links = []
    current_page = 1

    while current_page <= max_pages:
        # Handle pagination (if exists)
        if current_page == 1:
            page_url = category_url
        else:
            page_url = f"{category_url}page/{current_page}/"

        print(f"Fetching page {current_page}: {page_url}")

        try:
            response = requests.get(page_url, headers=HEADERS, timeout=30)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'lxml')

            # Find all article links (adjust selector based on actual HTML structure)
            # Looking for links that are likely section pages
            articles = soup.find_all('article') or soup.find_all('div', class_='post')

            page_sections = []
            for article in articles:
                # Find the main link to the section
                link_tag = article.find('a', href=True)
                if link_tag:
                    link = link_tag['href']
                    title = link_tag.get_text(strip=True)

                    # Only include links that seem to be Tripitaka sections
                    if 'thripitaka' in link.lower() or 'tripitaka' in link.lower():
                        section_links.append({
                            'url': link,
                            'title': title,
                            'section_name': clean_section_name(link)
                        })
                        page_sections.append(link)

            print(f"  Found {len(page_sections)} sections on page {current_page}")

            # Check if there's a next page
            next_page = soup.find('a', class_='next') or soup.find('a', text=re.compile(r'Next|›|»'))
            if not next_page or len(page_sections) == 0:
                break

            current_page += 1
            time.sleep(1)  # Be polite to the server

        except requests.RequestException as e:
            print(f"Error fetching page {current_page}: {str(e)}")
            break

    print(f"\nTotal sections found: {len(section_links)}")
    return section_links

def get_pdf_links_from_section(section_url):
    """
    Extract all Google Drive PDF links from a section page.
    """
    pdf_links = []

    try:
        response = requests.get(section_url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'lxml')

        # Find all Google Drive links
        all_links = soup.find_all('a', href=True)

        for link in all_links:
            href = link['href']
            if 'drive.google.com' in href:
                file_id = extract_google_drive_id(href)
                if file_id:
                    link_text = link.get_text(strip=True)
                    pdf_links.append({
                        'file_id': file_id,
                        'url': href,
                        'title': link_text or f'file_{file_id}'
                    })

        time.sleep(0.5)  # Be polite

    except requests.RequestException as e:
        print(f"Error fetching section {section_url}: {str(e)}")

    return pdf_links

## 5. Download Functions

In [7]:
def download_section_pdfs(section_info, pdf_links):
    """
    Download all PDFs for a given section.
    """
    section_name = section_info['section_name']
    section_dir = os.path.join(TRIPITAKA_DIR, section_name)
    os.makedirs(section_dir, exist_ok=True)

    print(f"\nDownloading section: {section_name}")
    print(f"Found {len(pdf_links)} PDFs")

    download_results = {
        'section': section_name,
        'total': len(pdf_links),
        'successful': 0,
        'failed': 0,
        'skipped': 0,
        'files': []
    }

    for idx, pdf_info in enumerate(tqdm(pdf_links, desc=f"Downloading {section_name}")):
        file_id = pdf_info['file_id']
        title = sanitize_filename(pdf_info['title'])

        # Generate filename
        if title.lower().endswith('.pdf'):
            filename = title
        else:
            filename = f"{title}_{file_id[:8]}.pdf"

        output_path = os.path.join(section_dir, filename)

        # Check if file already exists
        if os.path.exists(output_path):
            print(f"  Skipping (already exists): {filename}")
            download_results['skipped'] += 1
            download_results['files'].append({
                'filename': filename,
                'status': 'skipped',
                'file_id': file_id
            })
            continue

        # Download the file
        success = download_from_google_drive(file_id, output_path)

        if success and os.path.exists(output_path):
            download_results['successful'] += 1
            download_results['files'].append({
                'filename': filename,
                'status': 'success',
                'file_id': file_id,
                'size': os.path.getsize(output_path)
            })
        else:
            download_results['failed'] += 1
            download_results['files'].append({
                'filename': filename,
                'status': 'failed',
                'file_id': file_id
            })

        time.sleep(1)  # Rate limiting

    return download_results

## 6. Main Execution Pipeline

In [8]:
def main():
    """
    Main execution pipeline.
    """
    print("="*80)
    print("TRIPITAKA PDF DOWNLOADER")
    print("="*80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Output directory: {TRIPITAKA_DIR}\n")

    # Step 1: Get all section links
    print("Step 1: Fetching all Tripitaka sections...")
    section_links = get_section_links(BASE_URL, max_pages=3)  # Adjust max_pages as needed

    if not section_links:
        print("No sections found. Please check the website structure or URL.")
        return

    # Save section metadata
    sections_file = os.path.join(TRIPITAKA_DIR, '_sections_metadata.json')
    with open(sections_file, 'w', encoding='utf-8') as f:
        json.dump(section_links, f, indent=2, ensure_ascii=False)
    print(f"Section metadata saved to: {sections_file}")

    # Step 2: Process each section
    all_results = []

    for idx, section_info in enumerate(section_links, 1):
        print(f"\n{'='*80}")
        print(f"Section {idx}/{len(section_links)}: {section_info['title']}")
        print(f"URL: {section_info['url']}")
        print(f"{'='*80}")

        # Get PDF links from this section
        pdf_links = get_pdf_links_from_section(section_info['url'])

        if not pdf_links:
            print(f"No PDFs found in this section.")
            continue

        # Download PDFs
        results = download_section_pdfs(section_info, pdf_links)
        all_results.append(results)

        # Save progress after each section
        progress_file = os.path.join(TRIPITAKA_DIR, '_download_progress.json')
        with open(progress_file, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)

    # Step 3: Generate final report
    print("\n" + "="*80)
    print("DOWNLOAD SUMMARY")
    print("="*80)

    total_pdfs = sum(r['total'] for r in all_results)
    total_successful = sum(r['successful'] for r in all_results)
    total_failed = sum(r['failed'] for r in all_results)
    total_skipped = sum(r['skipped'] for r in all_results)

    print(f"Total sections processed: {len(all_results)}")
    print(f"Total PDFs found: {total_pdfs}")
    print(f"Successfully downloaded: {total_successful}")
    print(f"Failed: {total_failed}")
    print(f"Skipped (already exist): {total_skipped}")
    print(f"\nEnd time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # Save final report
    report_file = os.path.join(TRIPITAKA_DIR, '_download_report.json')
    final_report = {
        'timestamp': datetime.now().isoformat(),
        'summary': {
            'total_sections': len(all_results),
            'total_pdfs': total_pdfs,
            'successful': total_successful,
            'failed': total_failed,
            'skipped': total_skipped
        },
        'sections': all_results
    }

    with open(report_file, 'w', encoding='utf-8') as f:
        json.dump(final_report, f, indent=2, ensure_ascii=False)

    print(f"\nFinal report saved to: {report_file}")
    print("\nDownload complete!")

## 7. Run the Downloader

In [9]:
# Execute the main pipeline
main()

TRIPITAKA PDF DOWNLOADER
Start time: 2025-11-05 09:08:29
Output directory: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka

Step 1: Fetching all Tripitaka sections...
Fetching page 1: https://download.ifbcnet.org/category/thripitaka-books/
  Found 1 sections on page 1
Fetching page 2: https://download.ifbcnet.org/category/thripitaka-books/page/2/
  Found 0 sections on page 2

Total sections found: 1
Section metadata saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/_sections_metadata.json

Section 1/1: 
URL: https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b6%af%e0%b7%93%e0%b6%9d%e0%b6%b1%e0%b7%92%e0%b6%9a%e0%b7%8f%e0%b6%ba-thripitaka-digha-nikaya/


/tmp/ipython-input-2122405237.py:48: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  next_page = soup.find('a', class_='next') or soup.find('a', text=re.compile(r'Next|›|»'))



Found 3 PDFs


Downloading...
From (original): https://drive.google.com/uc?id=0B5RD8Fve5lhmeTNhUU8zQ0Z1LUE
From (redirected): https://drive.google.com/uc?id=0B5RD8Fve5lhmeTNhUU8zQ0Z1LUE&confirm=t&uuid=cea99bf2-7a14-4363-89c1-01a18d40852b
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/--thripitaka-digha-nikaya/දීඝ නිකාය 1_0B5RD8Fv.pdf

  0%|          | 0.00/140M [00:00<?, ?B/s]
  4%|▍         | 6.29M/140M [00:00<00:02, 57.8MB/s]
 12%|█▏        | 16.8M/140M [00:00<00:01, 75.6MB/s]
 20%|██        | 28.3M/140M [00:00<00:01, 91.8MB/s]
 27%|██▋       | 37.7M/140M [00:00<00:01, 81.0MB/s]
 39%|███▊      | 54.0M/140M [00:00<00:00, 107MB/s] 
 47%|████▋     | 65.5M/140M [00:00<00:00, 107MB/s]
 61%|██████    | 84.9M/140M [00:00<00:00, 133MB/s]
 71%|███████   | 99.1M/140M [00:01<00:00, 91.4MB/s]
 80%|███████▉  | 112M/140M [00:01<00:00, 98.6MB/s] 
 88%|████████▊ | 123M/140M [00:01<00:00, 87.4MB/s]
100%|██████████| 140M/140M [00:01<00:00, 93.5MB/s]
Downloadin


DOWNLOAD SUMMARY
Total sections processed: 1
Total PDFs found: 3
Successfully downloaded: 3
Failed: 0
Skipped (already exist): 0

End time: 2025-11-05 09:08:45

Final report saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/_download_report.json

Download complete!


## 8. Post-Download Analysis (Optional)

In [ ]:
# Analyze the downloaded files
import glob

print("Analyzing downloaded files...\n")

sections = [d for d in os.listdir(TRIPITAKA_DIR) if os.path.isdir(os.path.join(TRIPITAKA_DIR, d))]

for section in sorted(sections):
    section_path = os.path.join(TRIPITAKA_DIR, section)
    pdfs = glob.glob(os.path.join(section_path, '*.pdf'))
    total_size = sum(os.path.getsize(pdf) for pdf in pdfs)

    print(f"Section: {section}")
    print(f"  Files: {len(pdfs)}")
    print(f"  Total size: {total_size / (1024*1024):.2f} MB")
    print()

## 9. Retry Failed Downloads (Optional)

In [ ]:
# Load the download report and retry failed downloads
report_file = os.path.join(TRIPITAKA_DIR, '_download_report.json')

if os.path.exists(report_file):
    with open(report_file, 'r', encoding='utf-8') as f:
        report = json.load(f)

    print("Retrying failed downloads...\n")

    for section_result in report['sections']:
        failed_files = [f for f in section_result['files'] if f['status'] == 'failed']

        if failed_files:
            print(f"Section: {section_result['section']}")
            print(f"  Retrying {len(failed_files)} failed downloads...")

            section_dir = os.path.join(TRIPITAKA_DIR, section_result['section'])

            for file_info in failed_files:
                output_path = os.path.join(section_dir, file_info['filename'])
                success = download_from_google_drive(file_info['file_id'], output_path)

                if success:
                    print(f"    ✓ {file_info['filename']}")
                else:
                    print(f"    ✗ {file_info['filename']}")

                time.sleep(2)
else:
    print("No report file found. Run the main download first.")